# Proyecto No. 1 — Clasificación de autores
### Bag of Words con unigramas

**Materia:** Métodos de Texto (NLP) &nbsp;|&nbsp; **Proyecto No. 1: Text Classification**

**Equipo 3**
*Lambda Remiel Heredia Pérez y Diego Octavio Pérez Urbina*

---

Este cuaderno identifica al autor de una obra entre cuatro escritores.
El material se recorre de principio a fin en la presentación frente al grupo.

**Contenido**

1. Objetivo
2. Corpus y ligas a los acervos
3. Preparación de los archivos
4. Carga y partición entrenamiento / prueba
5. Preprocesamiento
6. Diccionario de las 300 palabras más frecuentes
7. Clasificadores
8. Resultados con el diccionario de todos los textos
9. Comparación: diccionario calculado solo con el entrenamiento
10. Palabras que separan a los autores
11. Guion breve para la presentación


## 1. Objetivo

El sistema debe decir quién escribió cada obra del conjunto de prueba.
Los autores son:

- Jean-Paul Sartre
- Albert Camus
- Simone de Beauvoir
- Emil Cioran

La unidad de clasificación es la **obra completa**, no la oración ni el párrafo.
De cada autor se apartan **3 obras** para la prueba. El resto se usa para entrenar.

La representación es una **bolsa de palabras (Bag of Words) de unigramas**.
El diccionario tiene las **300 palabras más frecuentes en todos los textos**.


## 2. Corpus y ligas a los acervos

No hay un dataset en dominio público con las obras completas de los cuatro autores.
Estas son las ligas que consultamos:

- Albert Camus, obras en francés: [Les Classiques des sciences sociales (UQAM)](https://www.classiques.uqam.ca/classiques/camus_albert/camus_albert.html).
  Esa biblioteca indica que aplica la regla canadiense de dominio público y pide respetar la ley del país de quien descarga.
- Jean-Paul Sartre, ensayos y fragmentos, sobre todo en inglés: [Marxists Internet Archive](https://www.marxists.org/reference/archive/sartre/).
  No es un conjunto de obras completas en francés.
- Simone de Beauvoir, *The Ethics of Ambiguity* y partes de *The Second Sex*, en inglés: [marxists.org](https://www.marxists.org/reference/subject/ethics/de-beauvoir/ambiguity/).
- Emil Cioran, ficha de catálogo de *Précis de décomposition*: [BnF](https://catalogue.bnf.fr/ark:/12148/cb16512908s).
  No encontramos un acervo abierto con sus obras completas en francés.

En México el derecho de autor dura la vida del autor y 100 años más.
Camus murió en 1960, Sartre en 1980, Beauvoir en 1986 y Cioran en 1995.
Sus libros siguen protegidos. Este cuaderno no contiene las obras.
Coloca en `data/` un archivo `.txt` por obra, tomado de una copia que tengas derecho a usar.
Los archivos de esa carpeta no se suben a git.

Todos los textos deben estar en **francés**.
Las palabras funcionales en francés son del autor.
Una traducción al español también lleva el estilo del traductor.
Los libros tempranos de Cioran en rumano quedan fuera: si no, el idioma lo identifica solo.

No incluyas *Les Possédés* de Camus. Es una adaptación de Dostoievski, y el estilo no es solo de Camus. Si ese archivo está en la carpeta, el cuaderno lo omite.


## 3. Preparación de los archivos

Abre este cuaderno desde la carpeta `Proyecto1`, o desde la raíz del repositorio.
Crea esta estructura. Un archivo `.txt` es una obra. Codificación UTF-8.

```text
data/
  sartre/
  camus/
  beauvoir/
  cioran/
```

Cada autor necesita **al menos 4 obras**: las 3 de prueba y al menos 1 de entrenamiento.
Si falta un autor, una obra de prueba o el entrenamiento, el cuaderno se detiene y dice qué falta.

Los nombres de las obras de prueba deben coincidir con estas claves
(mayúsculas, acentos, espacios y guiones bajos no importan).
Si el nombre del archivo **contiene** la clave, también cuenta.
Si hay dos archivos que contienen la misma clave, gana el nombre exacto.

| Autor | Archivo de prueba | Obra |
|---|---|---|
| Sartre | `la_nausee.txt` | *La Nausée* |
| Sartre | `huis_clos.txt` | *Huis clos* |
| Sartre | `lexistentialisme_est_un_humanisme.txt` | *L'existentialisme est un humanisme* |
| Camus | `letranger.txt` | *L'Étranger* |
| Camus | `le_mythe_de_sisyphe.txt` | *Le Mythe de Sisyphe* |
| Camus | `caligula.txt` | *Caligula* |
| Beauvoir | `linvitee.txt` | *L'Invitée* |
| Beauvoir | `pour_une_morale_de_lambiguite.txt` | *Pour une morale de l'ambiguïté* |
| Beauvoir | `le_deuxieme_sexe.txt` | *Le Deuxième Sexe* (un tomo; el otro tomo puede ir a entrenamiento) |
| Cioran | `precis_de_decomposition.txt` | *Précis de décomposition* |
| Cioran | `syllogismes_de_lamertume.txt` | *Syllogismes de l'amertume* |
| Cioran | `de_linconvenient_detre_ne.txt` | *De l'inconvénient d'être né* |

Ejemplos de obras de entrenamiento, con el mismo formato de nombre:

- Sartre: `le_mur.txt`, `les_mouches.txt`, `les_mains_sales.txt`, `les_mots.txt`
- Camus: `la_peste.txt`, `la_chute.txt`, `lexil_et_le_royaume.txt`, `lhomme_revolte.txt`, `noces.txt`, `lete.txt`, `les_justes.txt`
- Beauvoir: `les_mandarins.txt`, `memoires_dune_jeune_fille_rangee.txt`, `la_force_de_lage.txt`, `pyrrhus_et_cineas.txt`
- Cioran (solo obras en francés): `la_tentation_dexister.txt`, `histoire_et_utopie.txt`, `la_chute_dans_le_temps.txt`, `le_mauvais_demiurge.txt`, `ecartement.txt`

**Dependencias:** `numpy`, `pandas`, `scikit-learn`, `matplotlib`.


In [1]:
from __future__ import annotations

import os
import re
import unicodedata
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.naive_bayes import MultinomialNB

AUTHORS = {
    "sartre": "Jean-Paul Sartre",
    "camus": "Albert Camus",
    "beauvoir": "Simone de Beauvoir",
    "cioran": "Emil Cioran",
}

TEST_WORKS = {
    "sartre": ["lanausee", "huisclos", "lexistentialismeestunhumanisme"],
    "camus": ["letranger", "lemythedesisyphe", "caligula"],
    "beauvoir": ["linvitee", "pourunemoraledelambiguite", "ledeuxiemesexe"],
    "cioran": ["precisdedecomposition", "syllogismesdelamertume", "delinconvenientdetrene"],
}

TEST_TITLES = {
    "sartre": {
        "lanausee": "La Nausée",
        "huisclos": "Huis clos",
        "lexistentialismeestunhumanisme": "L'existentialisme est un humanisme",
    },
    "camus": {
        "letranger": "L'Étranger",
        "lemythedesisyphe": "Le Mythe de Sisyphe",
        "caligula": "Caligula",
    },
    "beauvoir": {
        "linvitee": "L'Invitée",
        "pourunemoraledelambiguite": "Pour une morale de l'ambiguïté",
        "ledeuxiemesexe": "Le Deuxième Sexe",
    },
    "cioran": {
        "precisdedecomposition": "Précis de décomposition",
        "syllogismesdelamertume": "Syllogismes de l'amertume",
        "delinconvenientdetrene": "De l'inconvénient d'être né",
    },
}

SHORT_LABELS = {
    "Jean-Paul Sartre": "Sartre",
    "Albert Camus": "Camus",
    "Simone de Beauvoir": "Beauvoir",
    "Emil Cioran": "Cioran",
}

TOKEN_RE = re.compile(r"[a-zàâäéèêëïîôùûüçœæ]+")
VOCAB_SIZE = 300
MIN_WORKS = 4
LABELS = list(AUTHORS.values())


## 4. Carga y partición

Cada archivo `.txt` es una obra. La partición no es aleatoria:
las tres obras nombradas arriba van a **prueba** y las demás van a **entrenamiento**.
Así el resultado se puede explicar obra por obra en la presentación.


In [2]:
def find_data_dir() -> Path:
    # CLASIFICACION_DATA permite apuntar a otra carpeta sin mover el cuaderno.
    if os.environ.get("CLASIFICACION_DATA"):
        return Path(os.environ["CLASIFICACION_DATA"]).expanduser().resolve()
    here = Path.cwd().resolve()
    candidates = [here / "data", here / "Proyecto1" / "data", here.parent / "data"]
    for path in candidates:
        if path.is_dir():
            return path
    return here / "data"


def normalize_key(name: str) -> str:
    decomposed = unicodedata.normalize("NFKD", name)
    without_marks = "".join(char for char in decomposed if not unicodedata.combining(char))
    return re.sub(r"[^a-z0-9]+", "", without_marks.lower())


def tokenize(text: str) -> list[str]:
    normalized = unicodedata.normalize("NFKC", text).lower()
    return TOKEN_RE.findall(normalized)


def _append_work(works: list[dict], author_key: str, title: str, path: Path, split: str) -> bool:
    text = path.read_text(encoding="utf-8")
    tokens = tokenize(text)
    if not tokens:
        print(f"Se omite el archivo vacío: {path}")
        return False
    works.append(
        {
            "autor_id": author_key,
            "autor": AUTHORS[author_key],
            "titulo": title,
            "archivo": path.name,
            "particion": split,
            "tokens": tokens,
        }
    )
    return True


def read_works(data_dir: Path) -> list[dict]:
    if not data_dir.is_dir():
        raise FileNotFoundError(
            f"No existe la carpeta de textos: {data_dir}. "
            "Crea data/sartre, data/camus, data/beauvoir y data/cioran, con un archivo .txt por obra."
        )

    works: list[dict] = []
    problems: list[str] = []

    for author_key, author_name in AUTHORS.items():
        folder = data_dir / author_key
        files = sorted(path for path in folder.glob("*.txt")) if folder.is_dir() else []
        if author_key == "camus":
            kept = []
            for path in files:
                if "possedes" in normalize_key(path.stem):
                    print(f"Se omite {path.name}: es una adaptación de Dostoievski, no solo de Camus.")
                else:
                    kept.append(path)
            files = kept
        if len(files) < MIN_WORKS:
            problems.append(
                f"{author_name}: hay {len(files)} archivo(s) en {folder}. Se necesitan al menos {MIN_WORKS}."
            )
            continue

        stems = [(path, normalize_key(path.stem)) for path in files]
        used: set[Path] = set()
        missing_tests: list[str] = []
        author_works: list[dict] = []

        for test_key in TEST_WORKS[author_key]:
            exact = [path for path, stem in stems if stem == test_key and path not in used]
            partial = [path for path, stem in stems if test_key in stem and path not in used]
            chosen = exact or partial
            title = TEST_TITLES[author_key][test_key]
            if not chosen:
                missing_tests.append(title)
                continue
            path = chosen[0]
            used.add(path)
            if not _append_work(author_works, author_key, title, path, "prueba"):
                missing_tests.append(f"{title} (archivo vacío)")

        if missing_tests:
            problems.append(f"{author_name}: faltan obras de prueba: {', '.join(missing_tests)}.")
            continue

        train_count = 0
        for path, _stem in stems:
            if path in used:
                continue
            if _append_work(author_works, author_key, path.stem.replace("_", " "), path, "entrenamiento"):
                train_count += 1

        if train_count < 1:
            problems.append(f"{author_name}: no queda ninguna obra de entrenamiento.")
            continue

        works.extend(author_works)

    if problems:
        detail = "\n".join(f"- {line}" for line in problems)
        raise FileNotFoundError(
            "El corpus no está completo. Cada autor necesita las 3 obras de prueba "
            "y al menos 1 obra más para entrenar.\n" + detail
        )
    return works


data_dir = find_data_dir()
print(f"Carpeta de textos: {data_dir}")
works = read_works(data_dir)

corpus = pd.DataFrame(
    [
        {
            "autor": work["autor"],
            "obra": work["titulo"],
            "partición": work["particion"],
            "archivo": work["archivo"],
            "tokens": len(work["tokens"]),
        }
        for work in works
    ]
)
author_order = {name: index for index, name in enumerate(LABELS)}
corpus["_autor"] = corpus["autor"].map(author_order)
corpus["_particion"] = corpus["partición"].map({"entrenamiento": 0, "prueba": 1})
corpus = corpus.sort_values(["_autor", "_particion", "obra"]).drop(columns=["_autor", "_particion"])
corpus = corpus.reset_index(drop=True)
display(corpus)
print(
    f"Obras: {len(corpus)}. "
    f"Entrenamiento: {(corpus['partición'] == 'entrenamiento').sum()}. "
    f"Prueba: {(corpus['partición'] == 'prueba').sum()}."
)


Carpeta de textos: /Users/octavioperezurbina/Documents/iteso/Text-Mining/Proyecto1/data


FileNotFoundError: El corpus no está completo. Cada autor necesita las 3 obras de prueba y al menos 1 obra más para entrenar.
- Jean-Paul Sartre: hay 0 archivo(s) en /Users/octavioperezurbina/Documents/iteso/Text-Mining/Proyecto1/data/sartre. Se necesitan al menos 4.
- Albert Camus: hay 0 archivo(s) en /Users/octavioperezurbina/Documents/iteso/Text-Mining/Proyecto1/data/camus. Se necesitan al menos 4.
- Simone de Beauvoir: hay 0 archivo(s) en /Users/octavioperezurbina/Documents/iteso/Text-Mining/Proyecto1/data/beauvoir. Se necesitan al menos 4.
- Emil Cioran: hay 0 archivo(s) en /Users/octavioperezurbina/Documents/iteso/Text-Mining/Proyecto1/data/cioran. Se necesitan al menos 4.

## 5. Preprocesamiento

Decisiones, y por qué:

- Pasar a minúsculas y normalizar Unicode. Se conservan los acentos del francés.
- Un token es una secuencia de letras, incluidas `à â ä é è ê ë ï î ô ù û ü ç œ æ`.
- Se quitan números y puntuación. *L'homme* queda como `l` + `homme`. Esas partículas son útiles aquí.
- **No** se quitan las palabras funcionales (*stop words*). Las 300 palabras más frecuentes son, en su mayoría, esas palabras. En atribución de autoría son la señal.
- **No** se lematiza. El lema borra hábitos del autor (*était* y *être* no cuentan igual).

La función `tokenize` ya se aplicó al cargar cada obra.


## 6. Diccionario de las 300 palabras más frecuentes

La consigna pide las 300 palabras más frecuentes **en todos los textos**.
Por eso el diccionario principal cuenta unigramas en entrenamiento y en prueba.
No usa las etiquetas de autor. Sí usa las palabras de las obras de prueba para elegir el vocabulario.

Un libro largo aporta más conteos que uno corto. Así se lee “más frecuentes en todos los textos”.

Después se repite el mismo experimento con un diccionario calculado **solo en el entrenamiento**.
Esa segunda tabla responde si las obras de prueba cambiaron la lista de 300 palabras.


In [ ]:
def build_vocab(selected: list[dict], limit: int = VOCAB_SIZE) -> tuple[list[str], pd.DataFrame]:
    counts: Counter[str] = Counter()
    for work in selected:
        counts.update(work["tokens"])
    vocab = [word for word, _frequency in counts.most_common(limit)]
    table = pd.DataFrame(
        {
            "rango": range(1, len(vocab) + 1),
            "palabra": vocab,
            "frecuencia": [counts[word] for word in vocab],
        }
    )
    return vocab, table


def counts_matrix(selected: list[dict], vocab: list[str]) -> np.ndarray:
    index = {word: column for column, word in enumerate(vocab)}
    matrix = np.zeros((len(selected), len(vocab)), dtype=np.float64)
    for row, work in enumerate(selected):
        for token in work["tokens"]:
            column = index.get(token)
            if column is not None:
                matrix[row, column] += 1.0
    return matrix


def relative_matrix(selected: list[dict], counts: np.ndarray) -> np.ndarray:
    lengths = np.array([len(work["tokens"]) for work in selected], dtype=np.float64)
    lengths[lengths == 0] = 1.0
    return counts / lengths[:, None]


vocab_global, tabla_global = build_vocab(works)
if len(vocab_global) < VOCAB_SIZE:
    print(f"Aviso: el corpus solo tiene {len(vocab_global)} palabras distintas. El diccionario usa todas.")
else:
    print(f"Diccionario global: {len(vocab_global)} unigramas.")
display(tabla_global)


## 7. Clasificadores

Cada obra es una muestra. Hay dos modelos sobre la misma bolsa de palabras:

- **Multinomial Naive Bayes**, sobre los conteos crudos. Es el modelo habitual para una bolsa de palabras. Este es el resultado principal.
- **Regresión logística**, sobre los conteos divididos por la longitud del libro (número de tokens ya limpios). Así un libro largo no gana solo por ser largo.

La longitud es el total de tokens de la obra, no solo la suma de las 300 palabras del diccionario.


In [ ]:
def evaluate(selected: list[dict], vocab: list[str], dictionary_name: str) -> dict:
    counts = counts_matrix(selected, vocab)
    relative = relative_matrix(selected, counts)
    labels = np.array([work["autor"] for work in selected])
    train_mask = np.array([work["particion"] == "entrenamiento" for work in selected])
    test_mask = ~train_mask

    naive_bayes = MultinomialNB()
    naive_bayes.fit(counts[train_mask], labels[train_mask])
    nb_pred = naive_bayes.predict(counts[test_mask])

    logistic = LogisticRegression(max_iter=5000, random_state=0)
    logistic.fit(relative[train_mask], labels[train_mask])
    lr_pred = logistic.predict(relative[test_mask])

    y_test = labels[test_mask]
    test_works = [work for work, is_test in zip(selected, test_mask) if is_test]
    detail = pd.DataFrame(
        [
            {
                "autor real": work["autor"],
                "obra": work["titulo"],
                "Naive Bayes": pred_nb,
                "acierto NB": pred_nb == work["autor"],
                "regresión logística": pred_lr,
                "acierto RL": pred_lr == work["autor"],
            }
            for work, pred_nb, pred_lr in zip(test_works, nb_pred, lr_pred)
        ]
    )
    summary = pd.DataFrame(
        [
            {
                "modelo": "Multinomial Naive Bayes",
                "diccionario": dictionary_name,
                "accuracy": accuracy_score(y_test, nb_pred),
                "f1_macro": f1_score(y_test, nb_pred, average="macro", labels=LABELS, zero_division=0),
            },
            {
                "modelo": "Regresión logística",
                "diccionario": dictionary_name,
                "accuracy": accuracy_score(y_test, lr_pred),
                "f1_macro": f1_score(y_test, lr_pred, average="macro", labels=LABELS, zero_division=0),
            },
        ]
    )
    return {
        "summary": summary,
        "detail": detail,
        "reports": {
            "nb": classification_report(y_test, nb_pred, labels=LABELS, digits=3, zero_division=0),
            "lr": classification_report(y_test, lr_pred, labels=LABELS, digits=3, zero_division=0),
        },
        "matrices": {
            "nb": confusion_matrix(y_test, nb_pred, labels=LABELS),
            "lr": confusion_matrix(y_test, lr_pred, labels=LABELS),
        },
        "relative_train": relative[train_mask],
        "y_train": labels[train_mask],
        "vocab": vocab,
    }


def plot_matrices(matrices: dict, heading: str) -> None:
    short = [SHORT_LABELS[label] for label in LABELS]
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for axis, key, name in zip(axes, ("nb", "lr"), ("Naive Bayes", "Regresión logística")):
        image = axis.imshow(matrices[key], cmap="Blues")
        axis.set_xticks(range(len(short)), short, rotation=30, ha="right")
        axis.set_yticks(range(len(short)), short)
        axis.set_xlabel("Predicción")
        axis.set_ylabel("Autor real")
        axis.set_title(name)
        for row in range(len(LABELS)):
            for column in range(len(LABELS)):
                axis.text(column, row, str(matrices[key][row, column]), ha="center", va="center")
        fig.colorbar(image, ax=axis, fraction=0.046)
    fig.suptitle(heading)
    fig.tight_layout()
    plt.show()


## 8. Resultados con el diccionario de todos los textos

La matriz de confusión se lee por filas: la fila es el autor real y la columna es la predicción.
La diagonal son los aciertos. Accuracy es la proporción de obras de prueba bien clasificadas.
El F1 macro es el promedio del F1 de los cuatro autores, sin dejar que un autor pese más que otro.


In [ ]:
resultado_global = evaluate(works, vocab_global, "todos los textos")
display(resultado_global["summary"])
print("Naive Bayes")
print(resultado_global["reports"]["nb"])
print("Regresión logística")
print(resultado_global["reports"]["lr"])
display(resultado_global["detail"])
plot_matrices(resultado_global["matrices"], "Diccionario de todos los textos")


## 9. Comparación: diccionario solo de entrenamiento

Mismo preprocesamiento, mismos modelos y la misma partición.
Cambia únicamente de dónde salen las 300 palabras: aquí solo de las obras de entrenamiento.


In [ ]:
train_works = [work for work in works if work["particion"] == "entrenamiento"]
vocab_train, tabla_train = build_vocab(train_works)
solo_en_prueba = sorted(set(vocab_global) - set(vocab_train))
print(f"Diccionario de entrenamiento: {len(vocab_train)} palabras.")
print(f"Palabras del diccionario global que no están en el de entrenamiento: {len(solo_en_prueba)}.")
if solo_en_prueba:
    print(", ".join(solo_en_prueba))

resultado_train = evaluate(works, vocab_train, "solo entrenamiento")
comparacion = pd.concat(
    [resultado_global["summary"], resultado_train["summary"]],
    ignore_index=True,
)
display(comparacion)
display(resultado_train["detail"])
plot_matrices(resultado_train["matrices"], "Diccionario solo de entrenamiento")


## 10. Palabras que separan a los autores

Para cada autor se compara la frecuencia relativa media en sus obras de entrenamiento
con la media del resto de autores. La tabla muestra las palabras del diccionario global
con mayor diferencia a favor de ese autor. Sirven para explicar el resultado, no como un análisis literario completo.


In [ ]:
def distinctive_words(relative_train: np.ndarray, y_train: np.ndarray, vocab: list[str], k: int = 8) -> pd.DataFrame:
    frames = []
    for label in LABELS:
        mask = y_train == label
        mean_in = relative_train[mask].mean(axis=0)
        mean_out = relative_train[~mask].mean(axis=0)
        delta = mean_in - mean_out
        order = np.argsort(delta)[-k:][::-1]
        frames.append(
            pd.DataFrame(
                {
                    "autor": label,
                    "palabra": [vocab[index] for index in order],
                    "diferencia": delta[order],
                }
            )
        )
    return pd.concat(frames, ignore_index=True)


palabras = distinctive_words(
    resultado_global["relative_train"],
    resultado_global["y_train"],
    resultado_global["vocab"],
)
display(palabras.round({"diferencia": 5}))
